# Task 1.2 — Key Assumptions
**Paper:** DynaMMo — Li et al., KDD 2009


## Assumption 1: Linear Gaussian Latent Dynamics

**Assumption:**  
The hidden state `z_t` evolves according to a linear autoregressive process with additive Gaussian noise: `z_t = A·z_{t-1} + ε_t`, `ε_t ~ N(0, Q)`. Similarly, the observation model is linear with Gaussian noise: `x_t = C·z_t + δ_t`, `δ_t ~ N(0, R)`.

**Why the method needs it:**  
Linearity and Gaussianity together make the E-step of EM analytically tractable — the Kalman Filter and RTS Smoother produce exact closed-form posteriors `P(z_t | X)`. If the dynamics were nonlinear or non-Gaussian, these posteriors would be intractable and the entire EM learning procedure would break down.

**Violation scenario:**  
Human motion capture data with abrupt direction changes (e.g., a person suddenly jumping), stock price data during market crashes, or any system with switching regimes — in all these cases the transition is nonlinear and the Gaussian assumption severely underestimates the true dynamics, leading to poor imputation near the nonlinear transition.

**Paper reference:**  
Section 3, Equations (1) and (2) — the LDS model equations explicitly state the Gaussian noise distributions. The entire EM derivation in Section 3.1 relies on these assumptions.

---

## Assumption 2: Stationarity of Latent Dynamics

**Assumption:**  
The transition matrix `A` and noise covariances `Q` and `R` are assumed constant across all time steps — that is, the latent dynamics are time-homogeneous (stationary). The same `A` governs `z_t = A·z_{t-1}` for every `t ∈ {1, …, T}`.

**Why the method needs it:**  
The M-step of EM estimates a single `A` by averaging sufficient statistics over all timesteps: `A = [Σ_t z_t z_{t-1}^T][Σ_t z_{t-1} z_{t-1}^T]^{-1}`. This average is only meaningful if the true transition is the same at every timestep. If `A` changes over time, averaging destroys the temporal structure.

**Violation scenario:**  
Chlorine sensor readings in a water network during a contamination event — the dynamics before, during, and after the contamination event are fundamentally different. A single `A` fitted to all three phases would average over regime changes and produce poor imputation during the transition period.

**Paper reference:**  
Section 3.1, Equations (11)–(14) — the M-step updates for `A` and `Q` pool statistics across the entire sequence. Section 4 (Experiments) uses water sensor and motion capture data, both of which are approximately stationary over short windows.

---

## Assumption 3: Low-Dimensional Shared Latent Structure

**Assumption:**  
All `d` observed sequences are assumed to be driven by a small number `k ≪ d` of shared latent factors. The emission matrix `C ∈ ℝ^{d×k}` is assumed to capture the complete cross-sequence dependency structure.

**Why the method needs it:**  
The entire motivation for using a latent variable model is that the `d` sequences are not independent — they co-evolve. If there were no shared structure (i.e., each sequence were driven by its own independent latent process), the LDS with a single shared `z_t` would be no better than modelling each sequence independently. The missing value recovery gains come specifically from the cross-sequence information flow through `z_t`.

**Violation scenario:**  
A dataset of `d` completely unrelated time series (e.g., stock prices from unrelated industries combined with weather readings and traffic counts) — there is no shared latent factor driving them, so the DynaMMo model would learn a meaningless `C` matrix and the latent states would not capture any useful cross-sequence structure, making imputation no better than mean fill.

**Paper reference:**  
Section 1 (Introduction, co-evolving sequences motivation), Section 2 (Problem Definition), and Section 4.1 where the paper explicitly evaluates on datasets where co-evolution is expected (water network sensors, motion capture joints — both are physically co-evolving).

---

## Assumption 4: Missing Completely at Random (MCAR)

**Assumption:**  
The paper implicitly assumes that values are Missing Completely At Random (MCAR) — the probability of a value being missing is independent of its true value and of other observed values.

**Why the method needs it:**  
The E-step skips the Kalman update for missing observations by simply omitting their dimensions from the innovation computation. This is statistically valid only if the missingness mechanism is ignorable — i.e., not correlated with the true value. If missingness depends on the value (e.g., a sensor fails precisely when the reading is above a threshold), the model's imputed values will be systematically biased.

**Violation scenario:**  
A temperature sensor that disconnects specifically when readings exceed 40°C (sensor protection shutdown). All missing values correspond to extreme temperatures. DynaMMo, not knowing this, will impute missing values as moderate temperatures near the mean of the observed data, systematically underestimating true extreme events.

**Paper reference:**  
Section 3.2 — the imputation equation `x̂_{t,j} = [C·μ_t^s]_j` treats missing values as unobserved Gaussian variables, implicitly assuming MCAR. The paper's experiment in Section 4.1 artificially removes values uniformly at random, which enforces MCAR by construction.
